In [1]:
from src.ofx_dataprep import OfxDataPrep
from src.llm_finance import LLMFinance

Load credit card data and execute data preparation

In [2]:
ofx_dataprep = OfxDataPrep()
transactions = ofx_dataprep.read_and_prep_data(data_path = "data/cc_nubank_2026-01-12.ofx")

Classifying transactions

In [3]:
filtered_transactions = transactions.query('in_out == "out"')

# Group messages to get unique messages
trans_grouped = filtered_transactions.groupby('memo')['id'].apply(list).reset_index(name='id_list')
trans_grouped.shape

(71, 2)

In [5]:
# Classifying transactions
models = [
    "lama-3.3-70b-versatile", # best accuracy?
    "qwen/qwen3-32b", # strong classifier + solid multilingual
]

for model_name in models:
    llm_finance = LLMFinance(model_name=model_name)
    transactions_classified = llm_finance.classify_batch(trans_grouped['memo'].head())
    print(model_name)
    display(transactions_classified)

lama-3.3-70b-versatile


,memo,category
0,44511303paulodos,Erro: Error code: 404 - {'error': {'message': ...
1,61414418marines,Erro: Error code: 404 - {'error': {'message': ...
2,99Food - NuPay,Erro: Error code: 404 - {'error': {'message': ...
3,A Ideal Pagpoko,Erro: Error code: 404 - {'error': {'message': ...
4,Abc Refeicoes,Erro: Error code: 404 - {'error': {'message': ...


qwen/qwen3-32b


,memo,category
0,44511303paulodos,"<think>\nOkay, let's see. The user provided a ..."
1,61414418marines,"<think>\nOkay, let's see. The user provided a ..."
2,99Food - NuPay,"<think>\nOkay, let's see. The user provided a ..."
3,A Ideal Pagpoko,"<think>\nOkay, let's see. The user wants me to..."
4,Abc Refeicoes,"<think>\nOkay, let's see. The user wants me to..."


In [6]:
transactions_classified['category'].values

array(['<think>\nOkay, let\'s see. The user provided a list of transaction categories and rules for categorizing each transaction. The task is to determine the correct category for the given transaction name "44511303paulodos".\n\nFirst, I need to check each category rule in order. The categories and their corresponding keywords are:\n\n1. Assinaturas e Serviços: Cinemark, Wellhub, Netflix, Facebk, Openai, Ebn*Spotify\n2. Transporte: Uber, Top Sp, Buser\n3. Mercado: Extra Hiper, Nagumo, Minimercado\n4. Horti Fruti: Horti Fruti\n5. Compras Parceladas: Parcela\n6. Saúde: Drogasil\n7. Restaurantes: Black Dogs\n8. Telefone: Recarga de celular\n9. Posto de Gasolina\n\nLooking at the transaction name "44511303paulodos", I need to check if any of the keywords from the categories are present. Let\'s go through each one:\n\n- Assinaturas e Serviços: None of the specified keywords (Cinemark, Wellhub, Netflix, etc.) are present in "44511303paulodos".\n- Transporte: Uber, Top Sp, Buser – none of t